# Blending and Stacking

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Finished&color=brightgreen)
[![Source](https://img.shields.io/static/v1.svg?label=GitHub&message=Source&color=181717&logo=GitHub)](https://github.com/particle1331/ok-transformer/blob/master/docs/nb/fundamentals/blending-stacking.ipynb)
[![Stars](https://img.shields.io/github/stars/particle1331/ok-transformer?style=social)](https://github.com/particle1331/ok-transformer)

---


## Introduction

Ensembling techniques combine models to obtain better predictive performance than could be obtained from any single model. Combining models means that shortcomings of each single model get balanced out. Note that stacking and blending ensembles a diverse group of strong learners that are trained on the same task. In contrast to bagging which is also an ensembling technique that combines weak learners (e.g. Random Forests). We will be particularly interested in two ensembling techniques: **stacking** and **blending**.

In this notebook, we will implement a class for stacking machine learning models trained on prediction probabilities of base models trained on the same task. Note that the implementation is fairly general, so this stacking model can be used without too much modifications for base models trained on other tasks. We also create a class that implements blending. This simply finds the best coefficients for creating a weighted sum of probabilities. The stacking class will allow parallelism and warm-starting to speed-up development cycles over base models that take a lot of time and resources to train.

<br>

```{margin}
𝗔𝘁𝘁𝗿𝗶𝗯𝘂𝘁𝗶𝗼𝗻: This notebook builds on algorithms outlined in the ensembling chapter of {cite}`AAAMLP`.
```

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from copy import deepcopy
from functools import partial
from scipy.optimize import minimize
from joblib import Parallel, delayed
from typing import Any, List, Dict, Union
from matplotlib_inline import backend_inline

from xgboost import XGBClassifier


warnings.simplefilter(action='ignore')
backend_inline.set_matplotlib_formats("svg")

seed = 42
random.seed(seed)
np.random.seed(seed)

N_JOBS = -1
TEST_RUN = False
NUM_FOLDS = 5

## IMDb Reviews Dataset

The dataset consists of 50,000 IMDb movie reviews selected for sentiment analysis. The sentiment of reviews is binary, with rating less than 5 is set to a sentiment score of 0, and rating at least 7 is set to a sentiment score of 1. No individual movie has more than 30 reviews.

```bash
USER="ymanojkumar023"
DATASET="kumarmanoj-bag-of-words-meets-bags-of-popcorn"
DATA_DIR=./data
kaggle datasets download -d ${USER}/${DATASET} -p ${DATA_DIR}
unzip ${DATA_DIR}/${DATASET}.zip -d ${DATA_DIR}/${DATASET} > /dev/null
rm ${DATA_DIR}/${DATASET}.zip
```

```text
100%|██████████████████████████████████████| 52.4M/52.4M [00:16<00:00, 3.38MB/s]
````

### Train and test split

In [ ]:
file_path = './data/kumarmanoj-bag-of-words-meets-bags-of-popcorn/labeledTrainData.tsv'
data = pd.read_csv(file_path, sep='\t')
if TEST_RUN:
    data = data.sample(frac=0.03, random_state=seed)

print(data.head())
print(data.shape)
data.sentiment.value_counts()

**Remark.** Perhaps a better approach would be to group reviews for the same movie in either the train of the test set. But we don't have that data available. 

## Stacking models

The idea with stacking is that predictions of models can be used as features to train new models. In principle, this process can be repeated indefinitely to train further models. To avoid overfitting, the process incorporates cross-validation folds. This just makes sure that data does not get leaked in the further layers.  


**Stacking algorithm with cross-validation**

1. Let test dataset be $\boldsymbol{\mathsf X}$ and split train dataset into $k$ folds $({\boldsymbol{\mathsf X}}_0, {\mathsf y}_0), \ldots, ({\boldsymbol{\mathsf X}}_{k-1}, {\mathsf y}_{k-1}).$
3. For $0 \leq i \leq k-1$ and each base model ${\mathsf m}^q$ with $1 \leq q \leq Q$:
    - Fit model ${\mathsf m}^q$ on $\bigcup_{j\neq i} ({\boldsymbol{\mathsf X}}_j, {\mathsf y}_j)$ to get  ${\mathsf m}^q_{\neq i}$.
    - Predict probabilities ${{\mathsf p}^q_i} = {\mathsf m}^q_{\neq i}({\boldsymbol{\mathsf X}}_i)$ for class 1 using trained model.
4. Fit model ${\mathsf m}^q$ on $\bigcup_j ({\boldsymbol{\mathsf X}}_j, {\mathsf y}_j)$ and predict probabilities ${{\mathsf p}^q} = {\mathsf m}^q({\boldsymbol{\mathsf X}})$ for class 1.
3. Stack predict probabilities to get meta-datasets: 
    - $(\bar{\boldsymbol{\mathsf X}}_0, {\mathsf y}_0), \ldots, (\bar{\boldsymbol{\mathsf X}}_{k-1}, {\mathsf y}_{k-1})$ with $\bar{\boldsymbol{\mathsf X}}_i = \left[{{\mathsf p}^1_i} \mid {{\mathsf p}^2_i} \mid \ldots \mid {{\mathsf p}^{Q}_i} \right]$
    - $\bar{\boldsymbol{\mathsf X}} = \left[{\mathsf p}^1 \mid {\mathsf p}^2 \mid \ldots \mid {\mathsf p}^{Q} \right]$
4. Repeat for another set of models with the train and test meta-datasets.


Note that after generating meta-features, the models will be retrained on the complete training set and predict on the test set to create test meta-features. This makes the test set look like an extra fold, so that cross-validation performance should be a good predictor of test performance.

### Training base models

Splitting the dataset and creating stratified folds:

In [ ]:
train, test = train_test_split(
    data, 
    test_size=0.20, 
    shuffle=True, 
    random_state=seed
)

train = train.drop('id', axis=1).reset_index(drop=True)
test  = test .drop('id', axis=1).reset_index(drop=True)

skf = StratifiedKFold(n_splits=NUM_FOLDS)
train['fold'] = -1
for fold, (_, val_) in enumerate(skf.split(train.index, train.sentiment)):
    train.loc[val_, 'fold'] = fold

In [ ]:
train.head()

In [ ]:
test.head()

Note that we specify models with initializers instead of actual models:

In [ ]:
lr = lambda: make_pipeline(
    TfidfVectorizer(max_features=1000),
    LogisticRegression(random_state=seed)
)

lr_cnt = lambda: make_pipeline(
    CountVectorizer(),
    LogisticRegression(random_state=seed)
)

rf_svd = lambda: make_pipeline(
    TfidfVectorizer(max_features=None),
    TruncatedSVD(n_components=120, random_state=seed),
    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
)

Perform stacking algorithm described above:

In [ ]:
base_models = {
    'lr': lr, 
    'lr_cnt': lr_cnt, 
    'rf_svd': rf_svd
}

acc_score = lambda y_true, y_prob: accuracy_score(y_true, y_prob >= 0.5)
roc_score = lambda y_true, y_prob: roc_auc_score(y_true, y_prob)
train_ = train.copy()
test_ = test.copy()

cv_scores = {}
for q in base_models.keys():
    print(f"\nLevel 0 preds: {q}")
    rocs = []
    accs = []
    for i in range(NUM_FOLDS):
        train_fld = train.query(f'fold != {i}')
        train_oof = train.query(f'fold == {i}')

        # Create train meta-features
        model = base_models[q]()
        model.fit(
            X=train_fld.review.values, 
            y=train_fld.sentiment.values
        )
        p = model.predict_proba(train_oof.review.values)[:, 1]
        train.loc[train_oof.index, f"{q}_0"] = p

        # Compute cv scores
        roc = roc_score(train_oof.sentiment, p)
        acc = acc_score(train_oof.sentiment, p)
        rocs.append(roc)
        accs.append(acc)
        print(f"fold={i}, roc={roc:.4f}, acc={acc:.4f}")

    cv_scores[q] = {
        'roc': np.array(rocs),
        'acc': np.array(accs),
    }

    # Create test meta-features
    model = base_models[q]()
    model.fit(
        X=train.review.values, 
        y=train.sentiment.values
    )
    test[f"{q}_0"] = model.predict_proba(test.review.values)[:, 1]

### Probability features

Here we look at each model's prediction probability that a review has positive sentiment: 

In [ ]:
train.head()

In [ ]:
test.head()

### Stacking with XGBoost

Here we train an XGBoost model that uses prediction probabilities of the base models as features. This model learns patterns of agreement and disagreement of the three base models. Hence, it is called a **meta-model**. For comparison, we first compute the performance of each base model:

In [ ]:
results = pd.DataFrame(
    index=[f"{q}_0" for q in base_models.keys()] + ['xgb_1'],
    columns=['cv_acc', 'test_acc', 'cv_roc', 'test_roc']
)

for q in base_models.keys():
    results.loc[f"{q}_0", 'test_acc'] = round(acc_score(test.sentiment, test[f'{q}_0']), 4)
    results.loc[f"{q}_0", 'test_roc'] = round(roc_score(test.sentiment, test[f'{q}_0']), 4)
    results.loc[f"{q}_0", 'cv_acc'] = f"{cv_scores[q]['acc'].mean():.4f} ± {cv_scores[q]['acc'].std():.4f}"
    results.loc[f"{q}_0", 'cv_roc'] = f"{cv_scores[q]['roc'].mean():.4f} ± {cv_scores[q]['roc'].std():.4f}"

Predictions of the base models are specified in the `columns` variable:

In [ ]:
columns = [c for c in train.columns if '_0' in c]
xgb = lambda: XGBClassifier(seed=seed, eval_metric='logloss')
q = 'xgb'

print(f"\nLevel 1 preds: {q}")
rocs = []
accs = []
for i in range(NUM_FOLDS):
    train_fld = train.query(f'fold != {i}')
    train_oof = train.query(f'fold == {i}')

    # Create train meta-features
    model = xgb()
    model.fit(
        X=train_fld[columns].values, 
        y=train_fld.sentiment.values
    )
    
    p = model.predict_proba(train_oof[columns].values)[:, 1]
    train.loc[train_oof.index, f"{q}_1"] = p

    # Compute cv scores
    roc = roc_score(train_oof.sentiment, p)
    acc = acc_score(train_oof.sentiment, p)
    rocs.append(roc)
    accs.append(acc)
    print(f"fold={i}, roc={roc:.4f}, acc={acc:.4f}")

cv_scores[q] = {
    'roc': np.array(rocs),
    'acc': np.array(accs),
}

# Create test meta-features
model = xgb()
model.fit(
    X=train[columns].values, 
    y=train.sentiment.values
)
test[f"{q}_1"] = model.predict_proba(test[columns].values)[:, 1]

### Initial results

Our initial result with stacking. The stacked model has better test performance and within range of best cv-performance. This shows that stacking can significantly improve over single models.

In [ ]:
results.loc[f"{q}_1", 'test_acc'] = round(acc_score(test.sentiment, test[f'{q}_1']), 4)
results.loc[f"{q}_1", 'test_roc'] = round(roc_score(test.sentiment, test[f'{q}_1']), 4)
results.loc[f"{q}_1", 'cv_acc'] = f"{cv_scores[q]['acc'].mean():.4f} ± {cv_scores[q]['acc'].std():.4f}"
results.loc[f"{q}_1", 'cv_roc'] = f"{cv_scores[q]['roc'].mean():.4f} ± {cv_scores[q]['roc'].std():.4f}"

results

## Stacking with warm start

Note the modular nature of the algorithm allowed us to copy the code above with minimal change for the XGBoost metamodel. This allows us to collect all our work above in a single class, allowing automated training and inference. To indicate the number of rounds of stacking performed, we introduce **levels**. This is analogous to layers in a neural network. 

We also include **warm starting**. This means that we use earlier levels in a trained stacker and stack them with layers. Only the new layers will be trained. It adds some complexity to the implementation, but warm-starting is a desirable feature, especially when training base models are expensive and we want to perform rapid iteration only on top models.

In [ ]:
class StackingClassifier:
    """Implements model stacking with predict probabilities."""

    def __init__(self, 
            models: List[Dict[str, Any]], 
            metrics: Dict[str, Any] = None,
            verbose: int = 1,
            num_folds: int = 5,

            # Advanced usage
            warm_start_level = None,
            warm_start_stack = None,
        ):
        """
        models:
            Dictionary of name-model initializer pairs for each level. 
            Some constraints:
                1) Level-wise unique names 
                2) Single model at last level
                3) Each model implements .fit and .predict_proba

        verbose: 
            0 = not verbose, 1 = verbose

        num_folds: 
            No. of stratified cross-validation folds
        
        metrics: 
            Name-metric pairs for such that metric is a function of 
            (y_true, y_predict_proba). This defaults to ROC-AUC. If
            there is warm-starting, defaults to metrics used by ref
            stack.

        warm_start_level (int):
            Warm-starting refers to using levels of existing trained stack 
            (stack passed in `warm_start_stack`) before `warm_start_level`.
            This is assumed to be in `1 <= j < len(warm_start_stack.models)`.
        
        warm_start_stack: 
            Use models and features from this stack so that predictions 
            of single models can be easily combined. And earlier models 
            need not be retrained. Some constraints:
                1) Earlier levels have trained models
                2) Earlier levels have meta-features
                3) Passed `models` will be appended to model dict list up 
                to warm start level of `warm_start_stack`.
        """

        self.models_fn = models
        self.verbose = verbose
        self.num_folds = num_folds
        self.metrics = {'roc_score':  roc_auc_score} if metrics is None else metrics

        # Learned attributes
        self.cv_scores_ = {}
        self.trained_models_ = [{m: None for m in d.keys()} for d in models]
        self.train_ = pd.DataFrame()

        # Warm starting model passed IFF level is specified
        assert (warm_start_level is None) == (warm_start_stack is None)
        if warm_start_level is not None:
            self.warm_level = warm_start_level
            self.warm_stack = warm_start_stack

            # Ignore upper level meta-features of reference stack
            skip = set()
            for l in range(self.warm_level, len(self.warm_stack.models_fn)):
                for q in self.warm_stack.models_fn[l].keys():
                    skip.add(f"{q}_{l}")

            _ = self.warm_stack.train_
            self.train_ = _[[c for c in _.columns if c not in skip]]
            
            # Subset cv scores of reference stack to include
            for i in range(self.warm_level):
                for k in [_ for _ in self.warm_stack.cv_scores_.keys() if f"_{i}" in _]:
                    self.cv_scores_[k] = self.warm_stack.cv_scores_[k]
            
            # Inherit metrics if None, else use passed metrics
            self.metrics = self.warm_stack.metrics if metrics is None else metrics
            
            # base = warm stack models, top = passed models
            self.models_fn = self.warm_stack.models_fn[:self.warm_level] + self.models_fn
            self.trained_models_ = self.warm_stack.trained_models_[:self.warm_level] + self.trained_models_
            
            # Warm start model
            self.fit(
                self.train_.drop(['fold', 'target'], axis=1), 
                self.train_.target, 
                start_level=self.warm_level
            )
    
    def fit(self, X: Union[pd.DataFrame, np.array], y: np.array, start_level=0):
        """Iteratively fit base models with metafeatures.

        Some constraints on the input:
            1) X must not have column named 'fold'
            2) X must not have column named 'target'
            3) X must not have column named f'{q}_{l}' for q model name, and 0 <= l < len(models)
            4) The target vector y is binary (0 or 1).
        """

        # Validate data (read docstring above)
        train = pd.DataFrame(X)

        # Create folds
        skf = StratifiedKFold(n_splits=self.num_folds, shuffle=False)
        train['target'] = y
        train['fold'] = -1
        for fold, (_, val_) in enumerate(skf.split(train.index, y)):
            train.loc[val_, 'fold'] = fold
        
        # Create metafeatures for each level
        for level in range(start_level, len(self.models_fn)):
            base_models = self.models_fn[level]
            columns = [c for c in train.columns if f'_{level-1}' in str(c)] if level >= 1 else train.columns.drop(['fold', 'target'])
            for q, model_init in base_models.items():
                if self.verbose:
                    print(f"\nLevel {level} preds: {q}")
    
                self.cv_scores_[f"{q}_{level}"] = {metric:[] for metric in self.metrics.keys()}
                for i in range(self.num_folds):
                    train_fld = train.query(f'fold != {i}')
                    train_oof = train.query(f'fold == {i}')

                    # Create train meta-features
                    model = model_init()
                    model.fit(
                        X=train_fld[columns], 
                        y=train_fld.target
                    )
                    p = model.predict_proba(train_oof[columns])[:, 1]
                    train.loc[train_oof.index, f"{q}_{level}"] = p

                    # Compute cv scores
                    message = []
                    for metric, metric_fn in self.metrics.items():
                        score = metric_fn(train_oof.target, p)
                        self.cv_scores_[f"{q}_{level}"][metric].append(score)
                        message.append(f'{metric}={score:.4f}')
                    
                    if self.verbose:
                        print(f"fold={i}, {', '.join(message)}")

                # Fit on entire train for inference
                self.trained_models_[level][q] = model_init().fit(train[columns], train.target)

        # Save features table
        self.train_ = train

        return self


    def predict_proba(self, X: Union[pd.DataFrame, np.array]):
        """Return prediction probabilities for each row of the input."""

        # Predict with trained models on meta-features one level below.
        test = pd.DataFrame(X)
        for level, trained_models in enumerate(self.trained_models_):
            columns = [c for c in test.columns if f'_{level-1}' in str(c)] if level >= 1 else test.columns
            for q, model in trained_models.items():
                pred = model.predict_proba(test[columns])[:, 1]
                test[f"{q}_{level}"] = pred
        
        # Return prediction of top-most model.
        test_pred = test[f"{list(self.trained_models_[-1].keys())[0]}_{len(self.trained_models_)-1}"]
        return np.c_[1-test_pred, test_pred]

The vectorizers expect 1-dimensional vectors$,$ so we define a helper transformer:

In [ ]:
class ReviewColumnExtractor(BaseEstimator, ClassifierMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): 
        return X.review.values.reshape(-1)

lr = lambda: make_pipeline(
    ReviewColumnExtractor(),
    TfidfVectorizer(max_features=1000),
    LogisticRegression(random_state=seed)
)

lr_cnt = lambda: make_pipeline(
    ReviewColumnExtractor(),
    CountVectorizer(),
    LogisticRegression(random_state=seed)
)

rf_svd = lambda: make_pipeline(
    ReviewColumnExtractor(),
    TfidfVectorizer(max_features=None),
    TruncatedSVD(n_components=120, random_state=seed),
    RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
)

Fitting the stacked models:

In [ ]:
models = [
    {'lr': lr, 'lr_cnt': lr_cnt, 'rf_svd': rf_svd},
    {'xgb': lambda: XGBClassifier(seed=seed, eval_metric='logloss')}
]

metrics = {
    'roc': lambda y_true, y_prob: roc_auc_score(y_true, y_prob),
    'acc': lambda y_true, y_prob: accuracy_score(y_true, y_prob >= 0.5)
}

stack = StackingClassifier(models, metrics=metrics)
stack.fit(train[['review']], train.sentiment);

### Reproducing initial results

Generated probability features can be accessed as follows:

In [ ]:
stack.train_.head(3)

Checking if this reproduces previous implementation:

In [ ]:
train.head(3)

CV-scores for AUC:



In [ ]:
pd.DataFrame({model: stack.cv_scores_[model]['roc'] for model in stack.cv_scores_.keys()}).head(3)

Checking if this reproduces previous results:

In [ ]:
pd.DataFrame({model: cv_scores[model]['roc'] for model in cv_scores.keys()}).head(3)

Looks good! So it looks like we didn't make any mistakes in our class implementation. For convenience, we define a helper function for updating results for new stacks. We will use this to update our initial results.

In [ ]:
def update_results(results, stack):
    """Helper function for viewing performance of various stacks."""

    X_test = test[['review']]
    y_test = test['sentiment']

    if results is None:
        results = pd.DataFrame()

    # Get name of final model
    l = len(stack.trained_models_) - 1
    q = list(stack.trained_models_[-1].keys())[0]

    # Iterate over metrics attached to stacker
    for metric, metric_fn in stack.metrics.items():
        
        # Compute cv scores
        cv_scores = np.array(stack.cv_scores_[f"{q}_{l}"][metric])
        results.loc[f"{q}_{l}", f'cv_{metric}'] = f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}"
        
        # Compute test scores
        y_pred = stack.predict_proba(X_test)[:, 1]
        score = metric_fn(y_test, y_pred)
        results.loc[f"{q}_{l}", f'test_{metric}'] = round(score, 4)

    return results


results = update_results(results, stack)
results

## Blending

A simple method of combining predictions of multiple models is to just take the weighted average of their predictions for some coefficients. This technique is aptly called **blending**. Note that the coefficients can be hand picked, but as we will show below, they can also be learned.

For the sake of comparison, we will be using the same base models that we used above. Although in actual practice, we would spend as much time as possible to improving the base models, and only use stacking for marginal gains. 

Blending works best when the base probabilities uncorrelated:

In [ ]:
stack.train_[['lr_0', 'lr_cnt_0', 'rf_svd_0']].corr()

Looks fairly uncorrelated. Let's try to blend the probabilities using some hand-designed coefficients.

In [ ]:
# AUC is scale invariant, so we dont bother dividing by total weights
metafeatures = stack.train_[['lr_0', 'lr_cnt_0', 'rf_svd_0']]
avg_preds = (metafeatures * [1, 1, 1]).sum(axis=1)
wtd_preds = (metafeatures * [1, 3, 1]).sum(axis=1)
rank_avg_preds = (metafeatures.rank() * [1, 1, 1]).sum(axis=1)
rank_wtd_preds = (metafeatures.rank() * [1, 3, 1]).sum(axis=1)

print(f"auc (train) (averaged):       {roc_auc_score(train.sentiment.values, avg_preds):.4f}")
print(f"auc (train) (wtd. avg):       {roc_auc_score(train.sentiment.values, wtd_preds):.4f}")
print(f"auc (train) (rank avg):       {roc_auc_score(train.sentiment.values, rank_avg_preds):.4f}") 
print(f"auc (train) (wtd. rank avg):  {roc_auc_score(train.sentiment.values, rank_wtd_preds):.4f}")

### Optimizing AUC

Since these coefficients are hand-designed, we may want to devise a strategy for automatically finding the optimal coefficients for blending. This is accomplished by the folowing class.

In [ ]:
class Blender:
    """Implement blending that maximizes AUC score."""
    
    def __init__(self, rank=False, random_state=42):
        self.coef_ = None
        self.rank = rank
        self.random_state = random_state

    def fit(self, X, y):
        """Find optimal blending coefficients."""
        
        X = pd.DataFrame(X)
        if self.rank:
            X = X.rank()

        self.coef_ = self._optimize_auc(X, y)
        return self

    def predict_proba(self, X):
        """Return blended probabilities for class 0 and class 1."""
        
        X = pd.DataFrame(X)
        if self.rank:
            X = X.rank()
            
        pred = np.sum(X * self.coef_, axis=1)
        return np.c_[1 - pred, pred]

    def _auc(self, coef, X, y):
        """Calculate AUC of blended predict probas."""

        auc = roc_auc_score(y, np.sum(X * coef, axis=1))
        return -1.0 * auc # min -auc = max auc
    
    def _optimize_auc(self, X, y):
        """
        Maximize AUC as a bound-constrained optimization problem using Nelder-Mead method 
        with coefficients initialized from a Dirichlet distribution with a = [1, ..., 1]. 
        
        Reference: 
        https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
        """
        partial_loss = partial(self._auc, X=X, y=y) 
        rng = np.random.RandomState(self.random_state)
        init_coef = rng.dirichlet(np.ones(X.shape[1]))
        return minimize(partial_loss, init_coef, 
                        method='Nelder-Mead', 
                        bounds=[(0, 1)]*X.shape[1])['x']

Calculating CV scores for blender model by warm-starting previous trained stack:

In [ ]:
blender_metrics = {'roc': roc_auc_score}

blender_stack = StackingClassifier(
    models=[{'blender': lambda: Blender()}],
    metrics=blender_metrics,
    warm_start_level=1,
    warm_start_stack=stack
)

Note that the metrics are inherited from the reference stack since we provided none. Also note that level 1 meta-features are not inherited by the warm-started stack. The inherited models are also only up to the warm start level.

In [ ]:
blender_stack.train_.head()

### Blending predict probabilities

Adding blending scores to results table. Blending has significantly better AUC scores but significantly worse accuracy scores than gradient boosting. 

In [ ]:
results = update_results(results, blender_stack)
results

Doing the same for blending with rank. Note that don't have an accuracy metric. This is because we only use this type of model for comparing scores.

In [ ]:
blender_rk_stack = StackingClassifier(
    models=[{'blender_rk': lambda: Blender(rank=True)}],
    metrics=blender_metrics,
    warm_start_level=1,
    warm_start_stack=stack
)

### Blending ranked predict probabilities

Blending ranked probabilities has even better cross-validation scores for AUC compared to blending raw probabilities. Recall that we observed the same phenomenon above for hand picked coefficients.

In [ ]:
results = update_results(results, blender_rk_stack)
results

**Remark.** If we assume that each fold has the same error distribution, then cross-validation performance should approximate the test AUC which can be thought of as just another fold of training data. Indeed, we can see this from the above results.

## Parallelizing Training

Generating features require training each model on each fold. This is very slow. Note that each training process are independent of each other as they only use static features from the previous level, so in principle can be easily parallelized. Note that we parallelize only the training on cross-validation folds.

Since joblib pickles every object used inside `Parallel`, we have to use stateless objects and be careful about shared memory. Results below show that there is significant speed up with parallelization using the `loky` backend.

In [ ]:
class StackingClassifierParallel:
    """Implements model stacking with predict probabilities."""

    def __init__(self, 
            models: List[Dict[str, Any]], 
            metrics: Dict[str, Any] = None,
            verbose: int = 1,
            num_folds: int = 5,
            backend: str = 'loky',
            n_jobs: int = 1,

            # Advanced usage
            warm_start_level = None,
            warm_start_stack = None,
        ):
        """
        models:
            Dictionary of name-model initializer pairs for each level. 
            Some constraints:
                1) Level-wise unique names 
                2) Single model at last level
                3) Each model implements .fit and .predict_proba

        verbose: 
            0 = not verbose, 1 = verbose

        num_folds: 
            No. of stratified cross-validation folds
        
        backend: 
            Default "loky". Backend for internal joblib.Parallel object.

        n_jobs: 
            No. of jobs passed to internal joblib.Parallel object.

        metrics: 
            Name-metric pairs for such that metric is a function of 
            (y_true, y_predict_proba). This defaults to ROC-AUC. If
            there is warm-starting, defaults to metrics used by ref
            stack.

        warm_start_level (int):
            Warm-starting refers to using levels of existing trained stack 
            (stack passed in `warm_start_stack`) before `warm_start_level`.
            This is assumed to be in `1 <= j < len(warm_start_stack.models)`.
        
        warm_start_stack: 
            Use models and features from this stack so that predictions 
            of single models can be easily combined. And earlier models 
            need not be retrained. Some constraints:
                1) Earlier levels have trained models
                2) Earlier levels have meta-features
                3) Passed `models` will be appended to model dict list up 
                to warm start level of `warm_start_stack`.
        """

        self.models_fn = models
        self.verbose = verbose
        self.num_folds = num_folds
        self.metrics = {'roc_score':  roc_auc_score} if metrics is None else metrics
        self.n_jobs = n_jobs
        self.backend = backend

        # Learned attributes
        self.cv_scores_ = {}
        self.trained_models_ = [{m: None for m in d.keys()} for d in models]
        self.train_ = pd.DataFrame()

        # Warm starting model passed IFF level is specified
        assert (warm_start_level is None) == (warm_start_stack is None)
        if warm_start_level is not None:
            self.warm_level = warm_start_level
            self.warm_stack = warm_start_stack

            # Ignore upper level meta-features of reference stack
            skip = set()
            for l in range(self.warm_level, len(self.warm_stack.models_fn)):
                for q in self.warm_stack.models_fn[l].keys():
                    skip.add(f"{q}_{l}")

            _ = self.warm_stack.train_
            self.train_ = _[[c for c in _.columns if c not in skip]]
            
            # Subset cv scores of reference stack to include
            for i in range(self.warm_level):
                for k in [_ for _ in self.warm_stack.cv_scores_.keys() if f"_{i}" in _]:
                    self.cv_scores_[k] = self.warm_stack.cv_scores_[k]
            
            # Inherit metrics if None, else use passed metrics
            self.metrics = self.warm_stack.metrics if metrics is None else metrics
            
            # base = warm stack models, top = passed models
            self.models_fn = self.warm_stack.models_fn[:self.warm_level] + self.models_fn
            self.trained_models_ = self.warm_stack.trained_models_[:self.warm_level] + self.trained_models_
            
            # Warm start model
            self.fit(
                self.train_.drop(['fold', 'target'], axis=1), 
                self.train_.target, 
                start_level=self.warm_level
            )
    
    def fit(self, X: Union[pd.DataFrame, np.array], y: np.array, start_level=0):
        """Iteratively fit base models with metafeatures.

        Some constraints on the input:
            1) X must not have column named 'fold'
            2) X must not have column named 'target'
            3) X must not have column named f'{q}_{l}' for q model name, and 0 <= l < len(models)
            4) The target vector y is binary (0 or 1).
        """


        # Create folds: add fold, target column to train
        train = self._create_kfolds(train=X, target=y, k=self.num_folds)
        
        # Create metafeatures for each level
        with Parallel(n_jobs=self.n_jobs, backend=self.backend, verbose=self.verbose) as parallel:
            for level in range(start_level, len(self.models_fn)):

                # Fit each current model on prev level meta-features
                base_models = self.models_fn[level]
                columns = self._get_metafeatures(train, level)

                for q, model_init in base_models.items():
                    if self.verbose:
                        print(f"\nLevel {level} preds: {q}")
        
                    # Parallel train on each fold
                    parallel_results = parallel(
                        delayed(self._predict_fold)(
                            train=train, 
                            model=model_init(), 
                            fold=i, 
                            columns=columns,
                            verbose=self.verbose
                        ) 
                        for i in range(self.num_folds)
                    )
                    
                    # Saving results to meta-features and cv scores
                    self.cv_scores_[f"{q}_{level}"] = {
                        metric: [0] * self.num_folds 
                        for metric in self.metrics.keys()
                    } 
                    for i in range(self.num_folds):
                        preds, scores = parallel_results[i]
                        train.loc[train.query(f'fold == {i}').index, f"{q}_{level}"] = preds
                        for metric in self.metrics.keys():
                            self.cv_scores_[f"{q}_{level}"][metric][i] = scores[metric]

                    # Fit on entire train for inference
                    self.trained_models_[level][q] = model_init().fit(train[columns], train.target)

        # Save meta-features table
        self.train_ = train

        return self


    def predict_proba(self, X: Union[pd.DataFrame, np.array]):
        """Return prediction probabilities for each row of the input."""

        # Predict with trained models on meta-features one level below.
        test = pd.DataFrame(X)
        for level, trained_models in enumerate(self.trained_models_):
            columns = [c for c in test.columns if f'_{level-1}' in str(c)] if level >= 1 else test.columns
            for q, model in trained_models.items():
                pred = model.predict_proba(test[columns])[:, 1]
                test[f"{q}_{level}"] = pred
        
        # Return prediction of top-most model.
        test_pred = test[f"{list(self.trained_models_[-1].keys())[0]}_{len(self.trained_models_)-1}"]
        return np.c_[1-test_pred, test_pred]


    def _get_metafeatures(self, train, level):
        """Get columns for meta-features of previous level models."""
        
        if level >= 1:
            return [c for c in train.columns if f'_{level-1}' in str(c)]
        else:
            return train.columns.drop(['fold', 'target'])


    def _create_kfolds(self, train, target, k):
        """Create stratified folds. Add fold and target column."""

        train = pd.DataFrame(train)
        skf = StratifiedKFold(n_splits=k, shuffle=False)
        train['target'] = target
        train['fold'] = -1
        for fold, (_, val_) in enumerate(skf.split(train.index, target)):
            train.loc[val_, 'fold'] = fold
        
        return train


    def _predict_fold(self, train, model, fold, columns, verbose):
        """Predict on cross-validation fold."""

        # Get folds; include target and feature cols.
        train_fld = train.query(f'fold != {fold}')
        train_oof = train.query(f'fold == {fold}')
        
        # Fit model.
        model.fit(train_fld[columns], train_fld.target)

        # Compute cv scores
        message = []
        scores = {}
        pred = model.predict_proba(train_oof[columns])[:, 1]
        
        for metric, metric_fn in self.metrics.items():
            score = metric_fn(train_oof.target, pred)
            scores[metric] = score
            message.append(f'{metric}={score:.4f}')
        
        if verbose:
            print(f"fold={fold}, {', '.join(message)}")

        # Return out-of-fold predictions.
        return pred, scores

Testing if the parallel implementation reproduces previous results:

In [ ]:
import os
os.environ['PYTHONWARNINGS']='ignore::FutureWarning'
os.environ['PYTHONWARNINGS']='ignore::UserWarning'

models = [
    {'lr': lr, 'lr_cnt': lr_cnt, 'rf_svd': rf_svd},
    {'xgb_pll': lambda: XGBClassifier(seed=seed, eval_metric='logloss')}
]

metrics = {
    'roc': lambda y_true, y_prob:  roc_auc_score(y_true, y_prob),
    'acc': lambda y_true, y_prob: accuracy_score(y_true, y_prob >= 0.5)
}

stack_pll = StackingClassifierParallel(models, metrics=metrics, n_jobs=N_JOBS)
stack_pll.fit(train[['review']], train.sentiment);

In [ ]:
results = update_results(results, stack_pll)

Here we have the opportunity to test warm starting. Note that it only starts training on the passed model which is stacked at level 1, over level 0 models of `stack_pll`.

In [ ]:
blender_rk_pll = StackingClassifierParallel(
    models=[{ 'blender_rk_pll': lambda: Blender(rank=True) }],
    metrics=blender_metrics,
    n_jobs=1,
    warm_start_level=1,
    warm_start_stack=stack_pll
)

### Equivalence of serial and parallel implementations

The results are the same for both training from scratch and warm-started stacked models. Thus, they perform the same computation. The only difference is that parallel implementation is faster as shown in the next section.

In [ ]:
results = update_results(results, blender_rk_pll)
results.loc[['blender_rk_1', 'blender_rk_pll_1', 'xgb_1', 'xgb_pll_1']]

### Parallel = faster

Now we would like to test if there is indeed a speedup and how much. First, we define more models for more levels. In particular, we create a new model which is a linear regression ranking model which we can used for classification.

In [ ]:
class LinearRegressionClassifier(BaseEstimator, ClassifierMixin):
    """
    Linear regression for model-based AUC optimization.
    Note that probabilities are transformed to rank scores.
    """
    
    def __init__(self): 
        self.lr = LinearRegression()
        
    def fit(self, X, y):
        self.lr.fit(pd.DataFrame(X).rank(), y)
        return self
        
    def predict_proba(self, X):
        return np.c_[[0]*len(X), self.lr.predict(pd.DataFrame(X).rank())]

Define the model dictionaries:

In [ ]:
level_0 = {
    'lr': lambda: make_pipeline(
        ReviewColumnExtractor(),
        TfidfVectorizer(max_features=1000),
        LogisticRegression(random_state=seed)
    ), 
    
    'lr_cnt': lambda: make_pipeline(
        ReviewColumnExtractor(),
        CountVectorizer(), 
        LogisticRegression(random_state=seed)
    ), 
}

level_1 = {
    'lr': lambda: LogisticRegression(random_state=seed),
    'linreg': lambda: make_pipeline(StandardScaler(), LinearRegressionClassifier()),
    'xgb': lambda: XGBClassifier(eval_metric="logloss", random_state=seed)
}

level_2 = {
    'linreg': lambda: make_pipeline(StandardScaler(), LinearRegressionClassifier()),
    'xgb': lambda: XGBClassifier(eval_metric="logloss", random_state=seed)
}

level_3 = {'blender': lambda: Blender(rank=True, random_state=seed)}

Define timing helper function:

In [ ]:
def time_training(model):
    """Return model training time vs percentage of train data."""
    
    X = pd.concat([train[['review']]]).reset_index(drop=True)
    y = list(train.sentiment.values)

    train_times = []
    N = len(X)
    for i in tqdm(range(10)):
        n = int(0.10 * (i + 1) * N)

        start_time = time.time()
        model.fit(X.iloc[:n], y[:n])
        end_time = time.time() - start_time

        train_times.append(end_time)
        time.sleep(7) # cooling down

    return train_times

Timing runs:

In [ ]:
models = [level_0, level_1, level_2, level_3]

serial = StackingClassifier(models, verbose=0, metrics=blender_metrics)
parallel = StackingClassifierParallel(models, verbose=0, metrics=blender_metrics, n_jobs=N_JOBS)

serial_times = time_training(serial)
parallel_times = time_training(parallel)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(10, 100+10, 10), serial_times, label="Serial")
plt.plot(range(10, 100+10, 10), parallel_times, label="Parallel")

plt.xlabel(f"Percentage of data (Total size = {len(train)})")
plt.ylabel(f"Training time (s)")
plt.legend()
plt.grid()

In [ ]:
(serial_times[-1] - serial_times[0]) / (parallel_times[-1] - parallel_times[0])

Training times seem to be linear in dataset size. We observe an improvement in training time with parallel processing by a fixed factor. Note that there is no overhead with parallelization, and we get a significant speed-up of more than 1.5x over serial!

### Four levels of stacking

In [ ]:
update_results(results, parallel).drop(['xgb_pll_1', 'blender_rk_pll_1'], axis=0)

This deep stack still has good cross-validation score (within the error range) and the best test AUC score. We can say that this improves upon the previous blending solution, i.e. blending rank probabilities of the three base classification models over vectorized words.